# Tax Calculator - Week 1 Features Testing

This notebook lets you interactively test the Week 1 foundation features:
- HTTP Fetcher
- Data Validator
- Cache Management
- Base Fetcher Classes

## Setup

First, let's import all the necessary modules:

In [1]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from taxcalc.fetchers.http_utils import HTTPFetcher, fetch_url
from taxcalc.fetchers.parsers.validators import TaxDataValidator
from taxcalc.cache.tax_data_cache import TaxDataCache, CacheEntry
from taxcalc.fetchers.base import BaseTaxDataFetcher, FetchResult, FetchStatus

import tempfile
from datetime import datetime, timedelta

print("✓ All modules imported successfully!")

✓ All modules imported successfully!


## 1. HTTP Fetcher Testing

Test the HTTP fetching utilities with retry logic:

In [2]:
# Create HTTP fetcher
fetcher = HTTPFetcher(
    timeout=10,
    retry_count=3,
    retry_delay=1.0
)

print(f"HTTP Fetcher Configuration:")
print(f"  Timeout: {fetcher.timeout}s")
print(f"  Retry count: {fetcher.retry_count}")
print(f"  User agent: {fetcher.user_agent}")

HTTP Fetcher Configuration:
  Timeout: 10s
  Retry count: 3
  User agent: TaxCalc-DataFetcher/1.0 (https://github.com/blackace007-cmd/tax-calculator)


In [3]:
# Test URL validation
test_urls = [
    'https://www.irs.gov/',
    'http://www.ftb.ca.gov/',
    'not a url',
    'ftp://invalid.com'
]

print("URL Validation Results:")
for url in test_urls:
    is_valid = fetcher.validate_url(url)
    status = '✓ Valid' if is_valid else '✗ Invalid'
    print(f"  {url:<40} {status}")

URL Validation Results:
  https://www.irs.gov/                     ✓ Valid
  http://www.ftb.ca.gov/                   ✓ Valid
  not a url                                ✗ Invalid
  ftp://invalid.com                        ✗ Invalid


In [ ]:
# Optional: Test actual fetch (uncomment to try)
# This will make a real HTTP request to IRS website

# try:
#     content = fetcher.fetch_html('https://www.irs.gov/')
#     print(f"Successfully fetched IRS homepage: {len(content)} characters")
#     print(f"Preview: {content[:200]}...")
# except Exception as e:
#     print(f"Fetch failed: {e}")

## 2. Data Validator Testing

Test the multi-level data validation system:

In [4]:
# Create validator for 2025
validator = TaxDataValidator(tax_year=2025)

# Valid federal tax data
valid_federal_data = {
    'tax_brackets': {
        'single': [
            (11925, 0.10),
            (48475, 0.12),
            (103350, 0.22),
            (197300, 0.24),
            (250525, 0.32),
            (626350, 0.35),
            (float('inf'), 0.37)
        ],
        'married_joint': [
            (23850, 0.10),
            (96950, 0.12),
            (206700, 0.22),
            (394600, 0.24),
            (501050, 0.32),
            (751600, 0.35),
            (float('inf'), 0.37)
        ],
        'married_separate': [
            (11925, 0.10),
            (48475, 0.12),
            (103350, 0.22),
            (197300, 0.24),
            (250525, 0.32),
            (375800, 0.35),
            (float('inf'), 0.37)
        ],
        'head_of_household': [
            (17000, 0.10),
            (64850, 0.12),
            (103350, 0.22),
            (197300, 0.24),
            (250500, 0.32),
            (626350, 0.35),
            (float('inf'), 0.37)
        ]
    },
    'standard_deductions': {
        'single': 15000,
        'married_joint': 30000,
        'married_separate': 15000,
        'head_of_household': 22500
    },
    'ltcg_brackets': {
        'single': [(48350, 0.00), (533400, 0.15), (float('inf'), 0.20)],
        'married_joint': [(96700, 0.00), (600050, 0.15), (float('inf'), 0.20)],
        'married_separate': [(48350, 0.00), (300025, 0.15), (float('inf'), 0.20)],
        'head_of_household': [(64750, 0.00), (566700, 0.15), (float('inf'), 0.20)]
    },
    'amt_exemption': {
        'single': 85700,
        'married_joint': 133300,
        'married_separate': 66650,
        'head_of_household': 85700
    },
    'amt_phaseout_start': {
        'single': 609350,
        'married_joint': 1218700,
        'married_separate': 609350,
        'head_of_household': 609350
    },
    'niit_thresholds': {
        'single': 200000,
        'married_joint': 250000,
        'married_separate': 125000,
        'head_of_household': 200000
    },
    'medicare_thresholds': {
        'single': 200000,
        'married_joint': 250000,
        'married_separate': 125000,
        'head_of_household': 200000
    }
}

# Validate
is_valid, errors = validator.validate_federal_data(valid_federal_data)

print(f"Validation Result: {'✓ VALID' if is_valid else '✗ INVALID'}")
if errors:
    print(f"Errors found: {len(errors)}")
    for error in errors:
        print(f"  - {error}")
else:
    print("No errors found!")

Validation Result: ✓ VALID
No errors found!


In [5]:
# Test with INVALID data (wrong bracket order)
invalid_data = {
    **valid_federal_data,
    'tax_brackets': {
        'single': [
            (50000, 0.10),
            (30000, 0.12),  # Wrong order!
            (float('inf'), 0.37)
        ],
        'married_joint': [(float('inf'), 0.10)],
        'married_separate': [(float('inf'), 0.10)],
        'head_of_household': [(float('inf'), 0.10)]
    }
}

is_valid, errors = validator.validate_federal_data(invalid_data)

print(f"Validation Result: {'✓ VALID' if is_valid else '✗ INVALID (expected)'}")
print(f"\nErrors found: {len(errors)}")
for error in errors:
    print(f"  - {error}")

Federal data validation failed with 1 errors


Validation Result: ✗ INVALID (expected)

Errors found: 1
  - federal/single: Bracket 1 upper limit $30,000 not greater than previous $50,000


## 3. Cache Management Testing

Test the local caching system:

In [6]:
# Create cache in temp directory
temp_dir = tempfile.mkdtemp()
cache = TaxDataCache(cache_dir=temp_dir)

print(f"Cache directory: {cache.cache_dir}")
print(f"Cache exists: {cache.cache_dir.exists()}")

Cache directory: /tmp/tmpbvwv8isx
Cache exists: True


In [7]:
# Save some test data
federal_data = {
    'tax_brackets': {
        'single': [(10000, 0.10), (50000, 0.20), (float('inf'), 0.37)]
    },
    'standard_deductions': {
        'single': 15000,
        'married_joint': 30000,
        'married_separate': 15000,
        'head_of_household': 22500
    }
}

ca_data = {
    'tax_brackets': {
        'single': [(10000, 0.01), (50000, 0.05), (float('inf'), 0.10)]
    },
    'sdi_rate': 0.012
}

# Save to cache
cache.save(2024, 'federal', federal_data, source_url='http://irs.gov/test')
cache.save(2024, 'california', ca_data, source_url='http://ftb.ca.gov/test')
cache.save(2025, 'federal', federal_data, source_url='http://irs.gov/test')

print("✓ Saved 3 cache entries")

✓ Saved 3 cache entries


In [8]:
# List cached years
years = cache.list_cached_years()
print(f"Cached years: {years}")

# Check what's cached
print("\nCache status:")
print(f"  2024 federal: {cache.is_cached(2024, 'federal')}")
print(f"  2024 california: {cache.is_cached(2024, 'california')}")
print(f"  2025 federal: {cache.is_cached(2025, 'federal')}")
print(f"  2025 california: {cache.is_cached(2025, 'california')}")

Cached years: [2024, 2025]

Cache status:
  2024 federal: True
  2024 california: True
  2025 federal: True
  2025 california: False


In [9]:
# Load cached data
entry = cache.load(2024, 'federal')

if entry:
    print(f"Loaded entry:")
    print(f"  Tax year: {entry.tax_year}")
    print(f"  Source: {entry.source}")
    print(f"  Data keys: {list(entry.data.keys())}")
    print(f"  Cached at: {entry.cached_at}")
    print(f"  Age: {entry.age_days():.4f} days")
    print(f"  Source URL: {entry.source_url}")
    print(f"  Is stale (90 days): {entry.is_stale(max_age_days=90)}")

Loaded entry:
  Tax year: 2024
  Source: federal
  Data keys: ['tax_brackets', 'standard_deductions']
  Cached at: 2025-10-30 21:37:24.403527
  Age: 0.0029 days
  Source URL: http://irs.gov/test
  Is stale (90 days): False


In [10]:
# Get detailed cache info
info = cache.get_cache_info(2024)

print("Cache info for 2024:")
for source, source_info in info['sources'].items():
    print(f"\n  {source}:")
    if source_info['cached']:
        print(f"    Cached: Yes")
        print(f"    Age: {source_info['age_days']:.4f} days")
        print(f"    Stale: {source_info['stale']}")
        print(f"    URL: {source_info['source_url']}")
    else:
        print(f"    Cached: No")

Cache info for 2024:

  federal:
    Cached: Yes
    Age: 0.0030 days
    Stale: False
    URL: http://irs.gov/test

  california:
    Cached: Yes
    Age: 0.0030 days
    Stale: False
    URL: http://ftb.ca.gov/test


In [11]:
# Get cache size
size_bytes = cache.get_cache_size()
print(f"Total cache size: {size_bytes:,} bytes ({size_bytes/1024:.2f} KB)")

Total cache size: 1,477 bytes (1.44 KB)


In [12]:
# Clean up temp cache
import shutil
shutil.rmtree(temp_dir)
print("✓ Temp cache cleaned up")

✓ Temp cache cleaned up


## 4. Base Fetcher Testing

Test the abstract base class framework:

In [13]:
# Create a simple concrete fetcher for testing
class DemoFetcher(BaseTaxDataFetcher):
    """Demo fetcher implementation"""
    
    def fetch(self, force: bool = False) -> FetchResult:
        self.log_fetch_start("http://demo.com")
        
        # Simulate fetching
        data = {
            'tax_brackets': {
                'single': [(10000, 0.10), (float('inf'), 0.20)]
            },
            'standard_deductions': {
                'single': 15000,
                'married_joint': 30000,
                'married_separate': 15000,
                'head_of_household': 22500
            }
        }
        
        result = self.create_result(
            data=data,
            status=FetchStatus.SUCCESS,
            source_url="http://demo.com"
        )
        
        result.add_warning("This is a demo fetcher")
        
        self.log_fetch_complete(result)
        return result
    
    def parse(self, raw_data: bytes) -> dict:
        return {'parsed': True}
    
    def validate(self, data: dict) -> tuple:
        return (True, [])

print("✓ DemoFetcher class created")

✓ DemoFetcher class created


In [14]:
# Create fetcher instance
fetcher = DemoFetcher(tax_year=2025, timeout=30, retry_count=3)

print(f"Fetcher: {fetcher}")
print(f"Tax year: {fetcher.tax_year}")
print(f"Timeout: {fetcher.timeout}s")
print(f"Retry count: {fetcher.retry_count}")
print(f"Source name: {fetcher.get_source_name()}")

Fetcher: DemoFetcher(tax_year=2025)
Tax year: 2025
Timeout: 30s
Retry count: 3
Source name: demo


In [15]:
# Fetch data
result = fetcher.fetch()

print(f"\nFetch Result:")
print(f"  Status: {result.status.value}")
print(f"  Success: {result.is_success()}")
print(f"  Tax year: {result.tax_year}")
print(f"  Source: {result.source}")
print(f"  Source URL: {result.source_url}")
print(f"  Fetched at: {result.fetched_at}")
print(f"  Data keys: {list(result.data.keys())}")
print(f"  Errors: {result.errors}")
print(f"  Warnings: {result.warnings}")

[demo] This is a demo fetcher



Fetch Result:
  Status: success
  Success: True
  Tax year: 2025
  Source: demo
  Source URL: http://demo.com
  Fetched at: 2025-10-30 21:42:32.846462
  Data keys: ['tax_brackets', 'standard_deductions']
  Errors: []
  Warnings: ['This is a demo fetcher']


In [16]:
# Test FetchStatus enum
print("FetchStatus values:")
for status in FetchStatus:
    print(f"  - {status.name}: {status.value}")

FetchStatus values:
  - SUCCESS: success
  - PARTIAL: partial
  - FAILED: failed
  - CACHED: cached


## 5. Integration Test

Test combining validator + cache together:

In [17]:
# Create cache and validator
temp_dir = tempfile.mkdtemp()
cache = TaxDataCache(cache_dir=temp_dir)
validator = TaxDataValidator(tax_year=2025)

# Sample data
test_data = valid_federal_data  # From earlier cell

# Validate
is_valid, errors = validator.validate_federal_data(test_data)
print(f"Data validation: {'✓ VALID' if is_valid else '✗ INVALID'}")

# If valid, cache it
if is_valid:
    cache.save(2025, 'federal', test_data, source_url='http://test.com')
    print("✓ Data cached successfully")
    
    # Load it back
    entry = cache.load(2025, 'federal')
    print(f"✓ Data loaded from cache (age: {entry.age_days():.4f} days)")
    
    # Verify integrity
    print(f"✓ Data integrity verified: {entry.data['standard_deductions']['single']} == {test_data['standard_deductions']['single']}")

# Cleanup
shutil.rmtree(temp_dir)
print("\n✓ Integration test complete!")

Data validation: ✓ VALID
✓ Data cached successfully
✓ Data loaded from cache (age: 0.0000 days)
✓ Data integrity verified: 15000 == 15000

✓ Integration test complete!


## Summary

You've tested all Week 1 foundation features:

1. ✓ **HTTP Fetcher** - URL validation and retry logic
2. ✓ **Data Validator** - Multi-level validation of tax data
3. ✓ **Cache Management** - Save/load with staleness detection
4. ✓ **Base Fetcher** - Abstract framework for implementing fetchers
5. ✓ **Integration** - Validator + Cache working together

Next steps:
- Week 2: Implement IRS Revenue Procedure fetcher
- Week 3: Implement California FTB fetcher
- Week 4: Build CLI tool and integration